# Qwen 7B baseline and reranked RAG

This notebook runs the five-question preview and then the full zero-shot, prompted, and reranked-RAG evaluation with Qwen2.5-7B-Instruct. It does not change the original 3B results.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'Select a T4 GPU in Runtime > Change runtime type'
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
from pathlib import Path
import os, subprocess, json, shutil
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/farmer-llm-7b')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
REPO = Path('/content/farmer-llm-project')
REPO_URL = 'https://github.com/sarthakmehra02/stuff.git'
if not REPO.exists(): subprocess.run(['git','clone',REPO_URL,str(REPO)],check=True)
else: subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
os.chdir(REPO)
os.environ['FARMER_MODEL_ID'] = 'Qwen/Qwen2.5-7B-Instruct'
print('Model:', os.environ['FARMER_MODEL_ID'])

In [ ]:
%pip install -r requirements-colab.txt
!grep -n 'rerank' src/run_conditions.py

In [ ]:
from src.rag import build_index
index, chunks = build_index()
print(f'Rebuilt index: {index.ntotal} vectors from {len(chunks)} chunks')

In [ ]:
%%time
!python src/run_conditions.py --conditions zero_shot,prompted,rag --limit 5 --out results/qwen7b_reranked --rerank --candidate-k 20
for condition in ['zero_shot','prompted','rag']:
    print('\n###', condition)
    with open(f'results/qwen7b_reranked/raw_{condition}.jsonl', encoding='utf-8') as f:
        for line in f: print(json.loads(line)['id'], json.loads(line)['answer'])

## Review q001–q005 before continuing

In [ ]:
%%time
!python src/run_conditions.py --conditions zero_shot,prompted,rag --out results/qwen7b_reranked --rerank --candidate-k 20 --resume
for path in Path('results/qwen7b_reranked').glob('raw_*.jsonl'): shutil.copy2(path, DRIVE_ROOT / path.name)
print('Copied Qwen 7B baseline outputs to', DRIVE_ROOT)